# **Proyecto IA**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Imports**

In [ ]:
# Standard libraries
import time

# Core numerical libs
import numpy as np

# Data & visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
# Model selection & evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, recall_score, f1_score, precision_score, roc_auc_score

# Preprocessing & pipelines
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

**Funciones auxiliares**

In [ ]:
# Separar los datos en entrenamiento, validación y test
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

### **Stroke**

**Cargar datos**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Proyecto IA - EOI/stroke.csv')

In [ ]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [ ]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [ ]:
# Balanceo de clases
df['stroke'].value_counts()

,count
stroke,
0,4861
1,249


In [ ]:
# Valores nulos
is_null = df.isna().any()
print("Columnas con NULOS:")
print(df.isna().sum()[df.isna().sum() > 0])

Columnas con NULOS:
bmi    201
dtype: int64


In [ ]:
# Rellenar datos nulos de BMI con la mediana (es más robusta que la media)
# f["bmi"] = df["bmi"].fillna(df["bmi"].median())
# creo que esto es mejor hacerlo después de separar entrenamiento/validación/test para evitar data leakage (si calculamos la mediana utilizando los datos que usaremos en validación o test, pasamos información de estos datos al set de entrenamiento)

In [ ]:
# Valores infinitos
is_inf = df.isin([np.inf, -np.inf]).any()
print("Columnas con INFINITOS:")
print(is_inf[is_inf])

Columnas con INFINITOS:
Series([], dtype: bool)


In [ ]:
# Valors duplicados
print("Columnas con DUPLICADOS:")
print(df.duplicated().sum())

Columnas con DUPLICADOS:
0


**Train/val/test split**


In [ ]:
train_set, val_set, test_set = train_val_test_split(df, stratify='stroke')

print("train:", train_set.shape, "| val:", val_set.shape, "| test:", test_set.shape)

train: (3066, 12) | val: (1022, 12) | test: (1022, 12)


In [ ]:
# Separar target y features
target = 'stroke'

# Para Entrenamiento
X_train = train_set.drop(columns=[target])
y_train = train_set[target]

# Para Validación
X_val = val_set.drop(columns=[target])
y_val = val_set[target]

# Para Test
X_test = test_set.drop(columns=[target])
y_test = test_set[target]

**Preprocesado**

In [ ]:
X_train.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
3135,33704,Male,44.0,1,0,Yes,Private,Rural,84.10,NaN,Unknown
3455,4297,Male,75.0,0,0,Yes,Govt_job,Urban,223.14,27.8,never smoked
1438,11232,Male,47.0,0,0,Yes,Private,Rural,93.55,31.4,never smoked
1566,18141,Male,76.0,0,1,Yes,Self-employed,Rural,101.43,29.1,Unknown
4182,27119,Female,28.0,0,0,No,Private,Rural,104.16,21.5,never smoked


In [ ]:
# Identificar automáticamente columnas numéricas y categóricas
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Pipeline para las variables NUMÉRICAS
num_pipeline = Pipeline([
    ('imputer_num', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Pipeline para las variables CATEGÓRICAS
cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# Transformador global
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
# Ajustar y transformar en el set de ENTRENAMIENTO
X_train_arr = preprocessor.fit_transform(X_train)

# Transformar el set de VALIDACIÓN usando los criterios de Train
X_val_arr = preprocessor.transform(X_val)

# Transformar el set de TEST usando los criterios de Train
X_test_arr = preprocessor.transform(X_test)

# Volver a convertir en DataFrame (ColumnTransformer devuelve array, que da igual para entrenar modelos pero así no podemos ver el head())
# Recuperar los nombres de las columnas generadas por el preprocessor
nombres_columnas = preprocessor.get_feature_names_out()

# Convertir los arrays a DataFrame
X_train_pre = pd.DataFrame(X_train_arr, columns=nombres_columnas, index=X_train.index)
X_val_pre = pd.DataFrame(X_val_arr, columns=nombres_columnas, index=X_val.index)
X_test_pre = pd.DataFrame(X_test_arr, columns=nombres_columnas, index=X_test.index)

In [ ]:
X_train_pre.head()

,num__id,num__age,num__hypertension,num__heart_disease,num__avg_glucose_level,num__bmi,cat__gender_Male,cat__gender_Other,cat__ever_married_Yes,cat__work_type_Never_worked,cat__work_type_Private,cat__work_type_Self-employed,cat__work_type_children,cat__Residence_type_Urban,cat__smoking_status_formerly smoked,cat__smoking_status_never smoked,cat__smoking_status_smokes
3135,-0.083004,-0.028571,1.0,0.0,-0.209891,0.000000,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3455,-0.871919,0.857143,0.0,0.0,3.647777,-0.021978,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1438,-0.685871,0.057143,0.0,0.0,0.052299,0.373626,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1566,-0.500520,0.885714,0.0,1.0,0.270930,0.120879,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4182,-0.259663,-0.485714,0.0,0.0,0.346674,-0.714286,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
X_val_pre.head()

,num__id,num__age,num__hypertension,num__heart_disease,num__avg_glucose_level,num__bmi,cat__gender_Male,cat__gender_Other,cat__ever_married_Yes,cat__work_type_Never_worked,cat__work_type_Private,cat__work_type_Self-employed,cat__work_type_children,cat__Residence_type_Urban,cat__smoking_status_formerly smoked,cat__smoking_status_never smoked,cat__smoking_status_smokes
2484,-0.051026,-0.800000,0.0,0.0,0.012347,1.648352,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2235,-0.054567,-0.342857,0.0,0.0,-0.300895,1.351648,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5101,0.002763,0.000000,0.0,0.0,0.174377,-0.384615,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
550,0.843536,-0.857143,0.0,0.0,0.979261,0.076923,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1889,-0.241420,0.771429,0.0,0.0,3.262676,1.109890,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
X_test_pre.head()

,num__id,num__age,num__hypertension,num__heart_disease,num__avg_glucose_level,num__bmi,cat__gender_Male,cat__gender_Other,cat__ever_married_Yes,cat__work_type_Never_worked,cat__work_type_Private,cat__work_type_Self-employed,cat__work_type_children,cat__Residence_type_Urban,cat__smoking_status_formerly smoked,cat__smoking_status_never smoked,cat__smoking_status_smokes
3009,-0.467710,-1.085714,0.0,0.0,-0.463481,-1.010989,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4211,-0.464115,1.028571,0.0,1.0,0.215718,-0.032967,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4250,0.230636,-0.057143,0.0,0.0,-0.939863,-0.351648,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4231,0.837982,0.228571,1.0,0.0,-0.220157,0.494505,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4005,-0.559862,-0.200000,0.0,0.0,0.201290,-0.274725,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


**Entrenar modelos de prueba**

In [ ]:
# Diccionario de modelos (activando balanceo de clases, ya que tenemos clases desbalanceadas)
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "SVC": SVC(class_weight='balanced', probability=True, random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=1, random_state=42)
}

In [ ]:
# Bucle de entrenamiento y evaluación
for nombre, modelo in modelos.items():
    print(f"====== {nombre} ======")

    # Entrenar con el set de entrenamiento
    modelo.fit(X_train_pre, y_train)

    # Predecir con el set de validación
    predicciones = modelo.predict(X_val_pre)
    probabilidades = modelo.predict_proba(X_val_pre)[:, 1]

    # Calcular tus métricas para la clase 1 (pacientes con riesgo de stroke)
    recall_1 = recall_score(y_val, predicciones, pos_label=1)
    precision_1 = precision_score(y_val, predicciones, pos_label=1, zero_division=0)
    f1_1 = f1_score(y_val, predicciones, pos_label=1)
    auc = roc_auc_score(y_val, probabilidades)

    # Mostrar resultado
    print(f"ROC-AUC:       {auc:.4f}")
    print(f"Recall (1):    {recall_1:.4f}")
    print(f"Precision (1): {precision_1:.4f}")
    print(f"F1-Score (1):  {f1_1:.4f}")

    print("\nMatriz de Confusión:")
    print(confusion_matrix(y_val, predicciones))
    print("=" * 40 + "\n")

====== Logistic Regression ======
ROC-AUC:       0.8375
Recall (1):    0.8000
Precision (1): 0.1246
F1-Score (1):  0.2156

Matriz de Confusión:
[[691 281]
 [ 10  40]]

====== Decision Tree ======
ROC-AUC:       0.5610
Recall (1):    0.1600
Precision (1): 0.1778
F1-Score (1):  0.1684

Matriz de Confusión:
[[935  37]
 [ 42   8]]

====== Random Forest ======
ROC-AUC:       0.7801
Recall (1):    0.0000
Precision (1): 0.0000
F1-Score (1):  0.0000

Matriz de Confusión:
[[972   0]
 [ 50   0]]

====== SVC ======
ROC-AUC:       0.7809
Recall (1):    0.6400
Precision (1): 0.1231
F1-Score (1):  0.2065

Matriz de Confusión:
[[744 228]
 [ 18  32]]

====== XGBoost ======
ROC-AUC:       0.7817
Recall (1):    0.1000
Precision (1): 0.3333
F1-Score (1):  0.1538

Matriz de Confusión:
[[962  10]
 [ 45   5]]



### **Altas ictus**

**Cargar datos**

In [ ]:
df2 = pd.read_csv('/content/drive/MyDrive/Proyecto IA - EOI/altas_ictus.csv', sep=';')

In [ ]:
df2.head()

,Fecha de ingreso,Fecha de alta,Día de la semana en la fecha del ingreso,Día de la semana en la fecha del alta,Zona Básica de Salud del Paciente,Ámbito de procedencia,Hospital,Área,Provincia,Edad,Sexo
0,2025-12-30T02:49:00+01:00,2025-12-30T21:51:00+01:00,Martes,Martes,ZBS Soria Rural,Rural,C.A. Soria,Soria,Soria,71,Hombre
1,2025-12-29T20:58:00+01:00,2025-12-31T15:21:00+01:00,Lunes,Miércoles,ZBS San Juan,Urbano,C.A.U. Salamanca,Salamanca,Salamanca,50,Hombre
2,2025-12-29T14:02:00+01:00,2025-12-31T12:28:00+01:00,Lunes,Miércoles,ZBS Eras de Renueva,Urbano,C.A.U. León,León,León,87,Mujer
3,2025-12-29T11:50:00+01:00,2025-12-31T16:59:00+01:00,Lunes,Miércoles,ZBS Aranda Norte,Urbano,C.A.U. Burgos,Burgos,Burgos,66,Hombre
4,2025-12-29T08:57:00+01:00,2025-12-29T15:37:00+01:00,Lunes,Lunes,ZBS Valladolid Rural II,Rural,H.U. Río Hortega,Valladolid Oeste,Valladolid,59,Mujer


In [ ]:
df2.describe()

,Edad
count,5362.000000
mean,75.931369
std,13.426058
min,2.000000
25%,67.000000
50%,78.000000
75%,86.000000
max,106.000000


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5362 entries, 0 to 5361
Data columns (total 11 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Fecha de ingreso                          5362 non-null   object
 1   Fecha de alta                             5362 non-null   object
 2   Día de la semana en la fecha del ingreso  5362 non-null   object
 3   Día de la semana en la fecha del alta     5362 non-null   object
 4   Zona Básica de Salud del Paciente         5362 non-null   object
 5   Ámbito de procedencia                     5362 non-null   object
 6   Hospital                                  5362 non-null   object
 7   Área                                      5362 non-null   object
 8   Provincia                                 5362 non-null   object
 9   Edad                                      5362 non-null   int64 
 10  Sexo                                      5362 n

In [ ]:
# Convertir fechas en zona horaria
df2["Fecha de ingreso"] = pd.to_datetime(
    df2["Fecha de ingreso"],
    utc=True
)

df2["Fecha de alta"] = pd.to_datetime(
    df2["Fecha de alta"],
    utc=True
)

# Calcular días de ingreso
df2["Dias ingreso"] = (
    df2["Fecha de alta"] - df2["Fecha de ingreso"]
).dt.days

In [ ]:
df2.head()

,Fecha de ingreso,Fecha de alta,Día de la semana en la fecha del ingreso,Día de la semana en la fecha del alta,Zona Básica de Salud del Paciente,Ámbito de procedencia,Hospital,Área,Provincia,Edad,Sexo,Dias ingreso
0,2025-12-30 01:49:00+00:00,2025-12-30 20:51:00+00:00,Martes,Martes,ZBS Soria Rural,Rural,C.A. Soria,Soria,Soria,71,Hombre,0
1,2025-12-29 19:58:00+00:00,2025-12-31 14:21:00+00:00,Lunes,Miércoles,ZBS San Juan,Urbano,C.A.U. Salamanca,Salamanca,Salamanca,50,Hombre,1
2,2025-12-29 13:02:00+00:00,2025-12-31 11:28:00+00:00,Lunes,Miércoles,ZBS Eras de Renueva,Urbano,C.A.U. León,León,León,87,Mujer,1
3,2025-12-29 10:50:00+00:00,2025-12-31 15:59:00+00:00,Lunes,Miércoles,ZBS Aranda Norte,Urbano,C.A.U. Burgos,Burgos,Burgos,66,Hombre,2
4,2025-12-29 07:57:00+00:00,2025-12-29 14:37:00+00:00,Lunes,Lunes,ZBS Valladolid Rural II,Rural,H.U. Río Hortega,Valladolid Oeste,Valladolid,59,Mujer,0


Framework Streamlit

In [1]:
!pip install streamlit
!streamlit hello &>/content/log.txt &
!streamlit run app.py &>/content/logs.txt &
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &
!pip install -r requirements.txt
!streamlit run app.py &>/content/logs.txt &


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 58.7 MB/s eta 0:00:00
Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Crear la aplicación

In [7]:
writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
# from preprocesado import add_bmi_missing # solo si usaste una función propia
@st.cache_resource
def cargar():
art = joblib.load("modelo.joblib")
return art["modelo"], art["umbral"]
modelo, UMBRAL = cargar()
st.title("Predicción de ictus")

st.title("PROYECTO IA - PREDICTOR DE ICTUS")

nombre = st.text_input("")

if nombre:
    st.success(f"")

Overwriting app.py


Ejecutar Streamlit en segundo plano

In [2]:
!streamlit run app.py &>/content/logs.txt &

Instalar Cloudflare Tunnel

In [12]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64    #MODIFICAR
!chmod +x cloudflared

Crear el enlace público

In [13]:
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501 #MODIFICAR

/bin/bash: line 1: ./cloudflared-linux-amd64: No such file or directory
